# Week 3: Prompts as Engineering Artifacts

## Purpose

This notebook evaluates version-controlled prompts as engineering artifacts. It develops an initial structured prompt for classifying patient-experience comments, defines a test suite with expected outputs, and measures prompt performance using:

- Exact-match accuracy for the classification label and confidence level
- Semantic similarity between the expected rationale and the model-generated rationale
- Input and output token usage

After evaluating the initial prompt, I identify a likely root cause of its classification errors and make one targeted prompt revision. I then compare the two prompt versions to determine whether the revision improved performance and to document any tradeoffs, such as changes in rationale similarity or token usage.

## HCAHPS Classification Task

The task uses comments related to the **Hospital Consumer Assessment of Healthcare Providers and Systems (HCAHPS)** survey, a standardized measure of patient experience in U.S. hospitals.

The labels in this notebook are simplified, derived categories based on selected HCAHPS-related topics. They are intended for this prompt-engineering experiment and do not represent the complete official HCAHPS survey categories.


In [1]:
# Imports
import os, json, pathlib
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import display, Markdown
import pandas as pd
from datetime import datetime
load_dotenv()

# === Connect to Foundry Project with LLM Deployment ===
project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

LIVE = os.getenv("FOUNDRY_PROJECT_ENDPOINT") is not None


# === Model Chat Function ===
def model_chat(messages, prompt, text_format=None, **kw):
    response = openai.responses.parse(
        model=prompt.model,
        instructions=prompt.text,
        input=messages,
        text_format=text_format,
        **prompt.params,
    )
    return response


print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [2]:
# Store Hugging Face model files in a project-local cache directory.
# This avoids downloading the model repeatedly and keeps the cache
# separate from the global user cache.
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())


from sentence_transformers import SentenceTransformer, util
# Load a small pretrained sentence-embedding model.
# It converts text into vectors so we can compare the meaning of
# expected rationales and model-generated rationales.

emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def exact_match(a, b):
    """Return 1.0 if the values match after ignoring capitalization and extra spaces."""
    return float(str(a).strip().lower() == str(b).strip().lower())

def semantic_similarity(a, b):
    """Compare the meaning of two texts using cosine similarity.

    Scores range from -1 to 1:

    - 1.0: the text vectors point in the same direction
    - 0.0: the text vectors have little similarity
    - -1.0: the text vectors point in opposite directions

    Sentence-embedding scores are often positive, so interpret results
    relative to the model and dataset rather than using universal cutoffs.
    Higher values indicate more similar meanings.
    """
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

c:\Users\brett\projects\github\bretttay24\cosc-650-applied-llm-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8912.08it/s]


metrics ready (exact-match + semantic)


## Part 1: Build the Structured Prompt
For this portion, I created version-controlled structured prompts in the "prompts\hcahps-classifier\" directory. The versioned prompts are stored as "v1.txt" and "v2.txt". I also created a custom prompt loader in "prompts\loader.py" that loads a versioned prompt and its parameters from "metadata.yaml". The "model_chat" function above uses those parameters, including temperature and max_output_tokens, when making the model call. This keeps prompt configuration separate from the notebook code.

### Define a response schema

In [3]:
from pydantic import BaseModel
from typing import Literal

HcahpsLabel = Literal[
    "communication_nurses",
    "communication_doctors",
    "staff_responsiveness",
    "communication_medicines",
    "discharge_information",
    "care_transition",
    "hospital_environment",
    "other",
]

Confidence = Literal["low", "medium", "high"]

class HcahpsClassification(BaseModel):
    label: HcahpsLabel
    rationale: str
    confidence: Confidence

### Load Version Controlled Prompt

In [4]:
# Load the custom loader for prompts
from prompts.loader import load

PROMPT_V1 = load(name="hcahps-classifier", version="v1")


## Part 2: Define Test Suite

### Test Suite: Input-Output Pairs

In [5]:
tests = [
    {
        "id": 1,
        "input": "My nurse noticed I was scared, sat down, and explained every test until I understood.",
        "expected": {
            "label": "communication_nurses",
            "rationale": "The nurse sat down and explained every test until the patient understood, which is clear nurse communication.",
            "confidence": "high",
        },
    },
    {
        "id": 2,
        "input": "The nurse kept interrupting me and walked out while I was still asking questions.",
        "expected": {
            "label": "communication_nurses",
            "rationale": "The nurse interrupted the patient and left while questions were being asked, a direct nurse-interaction concern.",
            "confidence": "high",
        },
    },
    {
        "id": 3,
        "input": "The doctor used so much medical jargon that I still did not know why I was admitted.",
        "expected": {
            "label": "communication_doctors",
            "rationale": "The doctor used medical jargon that left the patient unsure why they were admitted, showing unclear physician communication.",
            "confidence": "high",
        },
    },
    {
        "id": 4,
        "input": "Three doctors gave me three different answers about whether I needed surgery.",
        "expected": {
            "label": "communication_doctors",
            "rationale": "The patient received conflicting answers from three doctors about surgery, a physician communication issue.",
            "confidence": "high",
        },
    },
    {
        "id": 5,
        "input": "I pressed the call light because I could not reach the bathroom and waited nearly an hour.",
        "expected": {
            "label": "staff_responsiveness",
            "rationale": "The patient waited nearly an hour after using the call light for bathroom help, which concerns timeliness of assistance.",
            "confidence": "high",
        },
    },
    {
        "id": 6,
        "input": "Everyone was polite, but it took 45 minutes to bring my pain medicine after I asked.",
        "expected": {
            "label": "staff_responsiveness",
            "rationale": "Although staff were polite, the patient waited 45 minutes for requested pain medicine, so response time is the primary issue.",
            "confidence": "medium",
        },
    },
    {
        "id": 7,
        "input": "They started a new blood thinner but no one told me what it was for or what side effects to watch for.",
        "expected": {
            "label": "communication_medicines",
            "rationale": "Staff did not explain the new blood thinner's purpose or side effects during the stay, which is medication communication.",
            "confidence": "high",
        },
    },
    {
        "id": 8,
        "input": "The nurse gave me a new shot and said it was routine, but never explained why I needed it.",
        "expected": {
            "label": "communication_medicines",
            "rationale": "The nurse gave a new shot without explaining why it was needed, directly concerning medication communication.",
            "confidence": "high",
        },
    },
    {
        "id": 9,
        "input": "Before I left, no one explained what symptoms meant I should call the hospital once I was home.",
        "expected": {
            "label": "discharge_information",
            "rationale": "Before leaving, the patient was not told which symptoms should prompt a call after returning home, a discharge-information gap.",
            "confidence": "high",
        },
    },
    {
        "id": 10,
        "input": "They discharged me without telling my daughter what help I would need with bathing and meals.",
        "expected": {
            "label": "discharge_information",
            "rationale": "Staff did not explain the help needed with bathing and meals after discharge, which is discharge information.",
            "confidence": "high",
        },
    },
    {
        "id": 11,
        "input": "I left with six prescriptions and still do not understand what each one is supposed to do.",
        "expected": {
            "label": "care_transition",
            "rationale": "The patient left with six prescriptions without understanding each medication's purpose, showing they were unprepared to manage care at home.",
            "confidence": "high",
        },
    },
    {
        "id": 12,
        "input": "The discharge plan assumed I could climb stairs, even though I told them I live alone on the third floor.",
        "expected": {
            "label": "care_transition",
            "rationale": "The discharge plan ignored the patient's third-floor home and inability to climb stairs, so it did not account for their needs after leaving.",
            "confidence": "high",
        },
    },
    {
        "id": 13,
        "input": "The room was clean, but alarms and hallway conversations kept me awake all night.",
        "expected": {
            "label": "hospital_environment",
            "rationale": "Alarms and hallway conversations kept the patient awake, making noise in the hospital environment the central concern.",
            "confidence": "high",
        },
    },
    {
        "id": 14,
        "input": "My bed was uncomfortable, the room was freezing, and the bathroom smelled terrible.",
        "expected": {
            "label": "hospital_environment",
            "rationale": "The patient cites an uncomfortable bed, a freezing room, and a bad-smelling bathroom, all physical-environment issues.",
            "confidence": "high",
        },
    },
    {
        "id": 15,
        "input": "Parking was expensive and the food arrived cold every day.",
        "expected": {
            "label": "other",
            "rationale": "The comment concerns expensive parking and cold food, neither of which belongs to an HCAHPS composite domain listed here.",
            "confidence": "high",
        },
    },
    {
        "id": 16,
        "input": "Everybody was wonderful and I would recommend this hospital to my friends.",
        "expected": {
            "label": "other",
            "rationale": "This is general praise and a recommendation without a specific HCAHPS domain concern.",
            "confidence": "high",
        },
    },
    {
        "id": 17,
        "input": "The nurse was rude when I rang, then did not return with my pain pill for 45 minutes.",
        "expected": {
            "label": "staff_responsiveness",
            "rationale": "The comment mentions nurse rudeness, but the 45-minute delay in returning with pain medicine makes staff responsiveness the primary issue.",
            "confidence": "medium",
        },
    },
    {
        "id": 18,
        "input": "My doctor was kind, but I went home not knowing why I needed the new heart medicine.",
        "expected": {
            "label": "care_transition",
            "rationale": "Despite the kind doctor, the patient went home without understanding the new heart medicine's purpose, indicating inadequate preparation for self-care.",
            "confidence": "medium",
        },
    },
    {
        "id": 19,
        "input": "Nurse was nice i guess but nobody expained the new shot and my arm hurt after it.",
        "expected": {
            "label": "communication_medicines",
            "rationale": "Nobody explained the new shot, which is a medication-communication issue; the sore arm is secondary.",
            "confidence": "high",
        },
    },
    {
        "id": 20,
        "input": "I waited a long time for someone, but I cannot remember what I needed help with.",
        "expected": {
            "label": "staff_responsiveness",
            "rationale": "The patient reports waiting a long time for help, but cannot identify what help was needed, so the evidence supports responsiveness only weakly.",
            "confidence": "low",
        },
    },
]

### Test Suite: Testing Functions

In [6]:


def run_case(prompt, test_case):
    """
    Run a single test case against the given prompt using the model.

    Args:
        prompt: The prompt to use for the model.
        test_case: A dictionary containing the test case input.

    Returns:
        The parsed output from the model response.
    """
    response = model_chat(
        messages=[
            {"role": "user", "content": test_case["input"]}
        ],
        prompt=prompt,
        text_format=HcahpsClassification,
    )
    return response


def score(prompt, tests):
    """Score the model's performance on a set of test cases using the given prompt."""
    rows = []
    for test_case in tests:
        response = run_case(prompt, test_case)
        result = response.output_parsed
        expected = test_case["expected"]

        rows.append({
            "id": test_case["id"],
            "input": test_case["input"],
            "expected_label": expected["label"],
            "result_label": result.label,
            "expected_confidence": expected["confidence"],
            "result_confidence": result.confidence,
            "expected_rationale": expected["rationale"],
            "result_rationale": result.rationale,
            "label_exact": exact_match(expected["label"], result.label),
            "confidence_exact": exact_match(
                expected["confidence"], result.confidence
            ),
            "rationale_semantic_similarity": semantic_similarity(
                expected["rationale"], result.rationale
            ),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
        })

    raw_results = pd.DataFrame(rows)
    raw_results.attrs["prompt_name"] = prompt.name
    raw_results.attrs["prompt_version"] = prompt.version
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    results_dir = pathlib.Path("prompts") / prompt.name / "evaluation-results"

    results_dir.mkdir(parents=True, exist_ok=True)
    results_path = results_dir / f"{prompt.version}_eval_{timestamp}.csv"
    raw_results.to_csv(results_path, index=False)
    print(f"Saved raw results: {results_path}")

    label_accuracy = sum(row["label_exact"] for row in rows) / len(rows)
    confidence_accuracy = sum(row["confidence_exact"] for row in rows) / len(rows)
    average_semantic_similarity = sum(
        row["rationale_semantic_similarity"] for row in rows
    ) / len(rows)
    average_input_tokens = sum(row["input_tokens"] for row in rows) / len(rows)
    average_output_tokens = sum(row["output_tokens"] for row in rows) / len(rows)

    aggregate_results_df = pd.DataFrame(
        [{
            "prompt_name": prompt.name,
            "prompt_version": prompt.version,
            "evaluated_at": timestamp,
            "label_accuracy": label_accuracy,
            "confidence_accuracy": confidence_accuracy,
            "average_semantic_similarity": average_semantic_similarity,
            "average_input_tokens": average_input_tokens,
            "average_output_tokens": average_output_tokens,
            "test_case_count": len(raw_results),
        }]
    )
    aggregate_results_df.attrs["prompt_name"] = prompt.name
    aggregate_results_df.attrs["prompt_version"] = prompt.version

    aggregate_path = results_dir / f"{prompt.version}_agg_eval_results_{timestamp}.csv"
    aggregate_results_df.to_csv(aggregate_path, index=False)
    print(f"Saved aggregate results: {aggregate_path}")

    return (
        label_accuracy,
        confidence_accuracy,
        average_semantic_similarity,
        average_output_tokens,
        average_input_tokens,
        aggregate_results_df,
        raw_results,
    )


def display_results_table(
    label_acc,
    confidence_acc,
    avg_semantic_similarity,
    avg_output_tokens,
    avg_input_tokens,
    results_df,
):
    """Display the evaluation summary in a formatted table."""
    prompt_name = results_df.attrs["prompt_name"]
    prompt_version = results_df.attrs["prompt_version"]

    summary_table = pd.DataFrame(
        {
            "Metric": [
                "Label accuracy",
                "Confidence accuracy",
                "Average rationale semantic similarity",
                "Average output tokens",
                "Average input tokens",
            ],
            "Result": [
                f"{label_acc:.1%}",
                f"{confidence_acc:.1%}",
                f"{avg_semantic_similarity:.3f}",
                f"{avg_output_tokens:.1f}",
                f"{avg_input_tokens:.1f}",
            ],
        }
    )

    display(Markdown(f"### Prompt Evaluation: {prompt_name} ({prompt_version})"))
    display(summary_table)


def find_confidence_mismatches(results_df):
    """Return all non-token columns for cases with a confidence mismatch."""
    return results_df.loc[
        results_df["confidence_exact"] != 1.0
    ].drop(columns=["input_tokens", "output_tokens"]).copy()


def find_label_mismatches(results_df):
    """Return all non-token columns for cases with a label mismatch."""
    return results_df.loc[
        results_df["label_exact"] != 1.0
    ].drop(columns=["input_tokens", "output_tokens"]).copy()


def find_low_similarity_rows(results_df, similarity_threshold=0.5):
    """Return readable label and rationale comparisons below the threshold."""
    columns = [
        "id",
        "input",
        "expected_label",
        "result_label",
        "rationale_semantic_similarity",
        "expected_rationale",
        "result_rationale",
    ]
    return results_df.loc[
        results_df["rationale_semantic_similarity"] < similarity_threshold,
        columns,
    ].copy()


## Part 3: Evaluate Prompts and Show a Tradeoff 

### Run the Test Suite on the Versioned Prompts

#### Test Suite Evaluation: Prompt V1 

In [7]:
# === Score and unpack results of V1 prompt ===
(
    label_acc_v1,
    confidence_acc_v1,
    avg_semantic_similarity_v1,
    avg_output_tokens_v1,
    avg_input_tokens_v1,
    aggregate_results_df_v1,
    raw_results_df_v1,
) = score(PROMPT_V1, tests)

# === Display Aggregate Results ===
display_results_table(
    label_acc_v1,
    confidence_acc_v1,
    avg_semantic_similarity_v1,
    avg_output_tokens_v1,
    avg_input_tokens_v1,
    aggregate_results_df_v1,
)


Saved raw results: prompts\hcahps-classifier\evaluation-results\v1_eval_20260913T144552.csv
Saved aggregate results: prompts\hcahps-classifier\evaluation-results\v1_agg_eval_results_20260913T144552.csv


### Prompt Evaluation: hcahps-classifier (v1)

,Metric,Result
0,Label accuracy,90.0%
1,Confidence accuracy,80.0%
2,Average rationale semantic similarity,0.670
3,Average output tokens,40.5
4,Average input tokens,1089.8


##### V1 Label Mismatch

In [8]:
print("Label mismatches:")
display(find_label_mismatches(raw_results_df_v1).style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left",
    }
))

Label mismatches:


,id,input,expected_label,result_label,expected_confidence,result_confidence,expected_rationale,result_rationale,label_exact,confidence_exact,rationale_semantic_similarity
7,8,"The nurse gave me a new shot and said it was routine, but never explained why I needed it.",communication_medicines,communication_nurses,high,high,"The nurse gave a new shot without explaining why it was needed, directly concerning medication communication.","The comment focuses on the nurse's failure to explain the purpose of a new shot, which pertains to the interaction and communication with the nurse.",0.000000,1.000000,0.695000
11,12,"The discharge plan assumed I could climb stairs, even though I told them I live alone on the third floor.",care_transition,discharge_information,high,high,"The discharge plan ignored the patient's third-floor home and inability to climb stairs, so it did not account for their needs after leaving.","The comment discusses the discharge plan and its lack of consideration for the patient's living situation, which pertains to the information provided about post-hospital needs.",0.000000,1.000000,0.712000


##### V1 Confidence Mismatch

In [9]:
print("Confidence mismatches:")
display(find_confidence_mismatches(raw_results_df_v1).style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left",
    }
))

Confidence mismatches:


,id,input,expected_label,result_label,expected_confidence,result_confidence,expected_rationale,result_rationale,label_exact,confidence_exact,rationale_semantic_similarity
5,6,"Everyone was polite, but it took 45 minutes to bring my pain medicine after I asked.",staff_responsiveness,staff_responsiveness,medium,high,"Although staff were polite, the patient waited 45 minutes for requested pain medicine, so response time is the primary issue.","The comment emphasizes the delay in receiving pain medication after requesting it, which pertains to the timeliness of staff responsiveness.",1.000000,0.000000,0.690000
16,17,"The nurse was rude when I rang, then did not return with my pain pill for 45 minutes.",staff_responsiveness,staff_responsiveness,medium,high,"The comment mentions nurse rudeness, but the 45-minute delay in returning with pain medicine makes staff responsiveness the primary issue.","The comment highlights a delay in receiving pain medication after requesting it, which pertains to the timeliness of staff response.",1.000000,0.000000,0.692000
17,18,"My doctor was kind, but I went home not knowing why I needed the new heart medicine.",care_transition,care_transition,medium,high,"Despite the kind doctor, the patient went home without understanding the new heart medicine's purpose, indicating inadequate preparation for self-care.","The comment highlights the patient's lack of understanding regarding the purpose of a new medication upon discharge, which pertains to care transition.",1.000000,0.000000,0.501000
19,20,"I waited a long time for someone, but I cannot remember what I needed help with.",staff_responsiveness,staff_responsiveness,low,high,"The patient reports waiting a long time for help, but cannot identify what help was needed, so the evidence supports responsiveness only weakly.","The comment mentions waiting a long time for assistance, which aligns with the domain of staff responsiveness focusing on timeliness of help.",1.000000,0.000000,0.603000


##### V1 Semantic Similarity Between Expected and Returned Rationales < 0.5

In [10]:
print("Rationale similarity below 0.5:")
display(
    find_low_similarity_rows(raw_results_df_v1).style.set_properties(
        **{
            "white-space": "pre-wrap",
            "text-align": "left",
        }
    )
)

Rationale similarity below 0.5:


,id,input,expected_label,result_label,rationale_semantic_similarity,expected_rationale,result_rationale
15,16,Everybody was wonderful and I would recommend this hospital to my friends.,other,other,0.386000,This is general praise and a recommendation without a specific HCAHPS domain concern.,The comment expresses general praise for the hospital and staff without specifying a particular domain of experience.


#### Analysis of V1

The label mismatches occurred in difficult cases where either label could seem reasonable. However, according to the test definitions, the expected label was more appropriate than the model's returned label. For confidence, every mismatch occurred because the model predicted "high" when the expected confidence was lower. The saved V1 evaluation CSV also showed that the model predicted "high" for every test case. The lower rationale-similarity scores were interesting because the expected and returned rationales had similar meanings, but the returned rationales explicitly mentioned "staff" and "hospital", whereas the expected rationales referred more generally to HCAHPS domains.

#### Make a Change to Prompt V1 and Store as Prompt V2

Because I wanted to make only one change, I chose to address label classification rather than confidence classification or rationale similarity. I believe the root cause of the classification errors was ambiguity in the label definitions, which could cause the model to confuse similar labels. To address this, I added edge-case rules because the errors involved situations that could reasonably fit more than one category. Emphasizing the applicable rules should help the model distinguish between these cases.

My initial instinct was to clarify the label definitions themselves, but doing so could have changed the criteria used by the test suite and subtly altered the labels themselves. I decided that adding targeted edge-case guidance to the prompt was a more appropriate approach.

**Addition in Prompt:**
```text
## Edge-case rules
- If a comment concerns explaining a medication during the hospital stay,
  use communication_medicines even if it names a nurse or doctor.
- If a comment concerns understanding or managing medication after leaving
  the hospital, use care_transition.
- If a comment says the discharge plan ignored the patient's home situation,
  abilities, or stated needs, use care_transition rather than
  discharge_information.
```

#### Test Suite Evaluation: Prompt V2 

In [11]:
# === Load the custom loader for prompts for version 2 of the HCAHPS classifier ===
PROMPT_V2 = load(name="hcahps-classifier", version="v2")

# === Score the Prompt using Test Suite ===
(
    label_acc_v2,
    confidence_acc_v2,
    avg_semantic_similarity_v2,
    avg_output_tokens_v2,
    avg_input_tokens_v2,
    aggregate_results_df_v2,
    raw_results_df_v2,
) = score(PROMPT_V2, tests)

# === Display Results for Prompt Version 2 ===
display_results_table(
    label_acc_v2,
    confidence_acc_v2,
    avg_semantic_similarity_v2,
    avg_output_tokens_v2,
    avg_input_tokens_v2,
    aggregate_results_df_v2,
)

Saved raw results: prompts\hcahps-classifier\evaluation-results\v2_eval_20260913T144633.csv
Saved aggregate results: prompts\hcahps-classifier\evaluation-results\v2_agg_eval_results_20260913T144633.csv


### Prompt Evaluation: hcahps-classifier (v2)

,Metric,Result
0,Label accuracy,100.0%
1,Confidence accuracy,80.0%
2,Average rationale semantic similarity,0.667
3,Average output tokens,38.8
4,Average input tokens,1173.8


##### V2 Label Mismatch

In [12]:
print("Label mismatches:")
display(find_label_mismatches(raw_results_df_v2).style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left",
    }
))

Label mismatches:


,id,input,expected_label,result_label,expected_confidence,result_confidence,expected_rationale,result_rationale,label_exact,confidence_exact,rationale_semantic_similarity


##### V2 Confidence Mismatch

In [13]:
print("Confidence mismatches:")
display(find_confidence_mismatches(raw_results_df_v2).style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left",
    }
))

Confidence mismatches:


,id,input,expected_label,result_label,expected_confidence,result_confidence,expected_rationale,result_rationale,label_exact,confidence_exact,rationale_semantic_similarity
5,6,"Everyone was polite, but it took 45 minutes to bring my pain medicine after I asked.",staff_responsiveness,staff_responsiveness,medium,high,"Although staff were polite, the patient waited 45 minutes for requested pain medicine, so response time is the primary issue.","The comment emphasizes the delay in receiving pain medication after requesting it, which pertains to the timeliness of staff responsiveness.",1.000000,0.000000,0.690000
16,17,"The nurse was rude when I rang, then did not return with my pain pill for 45 minutes.",staff_responsiveness,staff_responsiveness,medium,high,"The comment mentions nurse rudeness, but the 45-minute delay in returning with pain medicine makes staff responsiveness the primary issue.","The comment emphasizes the delay in receiving pain medication after requesting it, which pertains to staff responsiveness.",1.000000,0.000000,0.754000
17,18,"My doctor was kind, but I went home not knowing why I needed the new heart medicine.",care_transition,care_transition,medium,high,"Despite the kind doctor, the patient went home without understanding the new heart medicine's purpose, indicating inadequate preparation for self-care.","The comment indicates the patient left the hospital without understanding the purpose of a prescribed medication, which pertains to care transition.",1.000000,0.000000,0.603000
19,20,"I waited a long time for someone, but I cannot remember what I needed help with.",staff_responsiveness,staff_responsiveness,low,high,"The patient reports waiting a long time for help, but cannot identify what help was needed, so the evidence supports responsiveness only weakly.","The comment mentions waiting a long time for assistance, which aligns with the domain of staff responsiveness.",1.000000,0.000000,0.583000


##### V2 Semantic Similarity Between Expected and Returned Rationales < 0.5

In [14]:
print("Rationale similarity below 0.5:")
display(
    find_low_similarity_rows(raw_results_df_v2).style.set_properties(
        **{
            "white-space": "pre-wrap",
            "text-align": "left",
        }
    )
)

Rationale similarity below 0.5:


,id,input,expected_label,result_label,rationale_semantic_similarity,expected_rationale,result_rationale
15,16,Everybody was wonderful and I would recommend this hospital to my friends.,other,other,0.357000,This is general praise and a recommendation without a specific HCAHPS domain concern.,The comment expresses general praise for the hospital and staff without specifying a particular domain of the hospital experience.


#### Tradeoff Analysis

As hoped, adding edge-case rules rather than revising the label definitions improved label-classification accuracy. I was concerned that changing the definitions could effectively alter the test criteria to match my results. To avoid that, I added targeted clarification for ambiguous edge cases instead. This change increased label accuracy from 90% to 100%.

However, the update did not address the confidence mismatches. The model predicted "high" confidence for every case, so when it missed a label or confidence value, it was confidently incorrect. Rationale semantic similarity was essentially unchanged, decreasing from 0.670 for V1 to 0.667 for V2. Input token usage also increased by fewer than 100 tokens. I consider the improvement in category-label accuracy worth the modest increase in input tokens while maintaining stable semantic similarity between the expected and returned rationales.

## Part 4: Find a Failure and Explain It

This prompt experiment showed no regressions on individual test cases. The targeted edge-case guidance improved label accuracy from 90% to 100% without meaningfully changing rationale semantic similarity, which decreased only from 0.670 for V1 to 0.667 for V2. Input token usage increased by fewer than 100 tokens. The prompt remained stable because the revision clarified specific ambiguous cases rather than changing the core label definitions or instructions.

However, the confidence label needs further improvement. The model predicted "high" confidence for every test case, including cases with incorrect labels or confidence values. This pattern limits the usefulness of the confidence field and is a shortcoming shared by both prompt versions.

## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).